# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Imports

In [ ]:
from _imports import *
from araras.ml.optuna.model_tools import plot_model_param_distribution

### 1.2. Policy

In [ ]:
POLICY = mixed_precision.Policy("mixed_float16")
mixed_precision.set_global_policy(POLICY)

### 1.3. Constants

In [ ]:
DATA_SEED = 99
TRAIN_SEED = 111

## 2. Data Loading and Preprocessing

In [ ]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

In [ ]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 3. Hyperparameters

In [ ]:
kparams = KParams.default()
kparams.learning_rate = 7e-5

## 4. Model Architectures

In [ ]:
def build_model(trial: optuna.Trial, kparams: dict, show_summary: bool = True) -> tf.keras.Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    from spektral.layers import GraphMasking, GlobalAvgPool

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    num_conv_layers = trial.suggest_int("num_conv_layers", 1, 4)

    for i in range(num_conv_layers):
        x = build_cnn1d(
            trial=trial,
            kparams=kparams,
            x=combined if i == 0 else x,  # Use combined only for the first layer
            name_prefix=f"conv1d_{i}",
            # Filters
            filters_range=trial.suggest_categorical(f"conv1d_{i}_filters", [64, 128, 256, 512]),
            # filters_step=40,
            # Kernel size
            kernel_size_range=(2, 12),
            kernel_size_step=2,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_{i}_strides", 1, 2),
            kernel_initializer=initializer,
        )
        pool_size = trial.suggest_int(f"pool_size_{i}", 2, 8, step=1)
        x = layers.MaxPooling1D(pool_size=pool_size, name=f"max_pool_{i}")(x)

    # ———————————————————————————————————— GNN ——————————————————————————————————— #
    A = build_knn_adjacency(rows=20, cols=200, k=trial.suggest_int("knn_k", 4, 16, step=4))

    x_graph, a_graph = GraphMasking()([combined, A])

    num_layers = trial.suggest_int("num_gnn_layers", 1, 4)
    for i in range(num_layers):
        # GNN layer with dropout
        y = build_cheb(
            trial=trial,
            kparams=kparams,
            x=x_graph if i == 0 else y,  # Use x_graph only for the first layer
            a_graph=a_graph,
            name_prefix=f"cheb_{i}",
            # Units
            units_range=(64, 256),
            units_step=32,
            # K
            K_range=(2, 5),
            K_step=1,
            # Dropout
            dropout_rate_range=(0.0, 0.5),
            dropout_rate_step=0.1,
            # Other parameters
            kernel_initializer=initializer,
        )

    # —————————————————————————————— Concat Branches ————————————————————————————— #
    # Concatenate CNN and GNN outputs
    x = layers.Flatten(name="flatten_cnn_output")(x)
    y = GlobalAvgPool(name="global_avg_pool")(y)
    y = layers.Flatten(name="flatten_gnn_output")(y)  # Mask is destroyed from this point

    x = layers.Concatenate(axis=-1, name="concat_cnn_gnn")([x, y])

    # ———————————————————————————— Extra dense layers ———————————————————————————— #
    num_dense_layers = trial.suggest_int("num_dense_layers", 0, 3)

    for i in range(num_dense_layers):
        # Dense layer with dropout
        x = build_dnn(
            trial=trial,
            kparams=kparams,
            x=x,
            name_prefix=f"dense_{i}",
            # Units
            units_range=(250, 600),
            units_step=50,
            # Dropout
            dropout_rate_range=(0.0, 0.4),
            dropout_rate_step=0.2,
            # Other parameters
            kernel_initializer=initializer,
        )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=kparams.get_optimizer(trial),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=False,  # Disable XLA JIT compilation
    )

    return model

## 5. Optuna Ask

In [ ]:
plot_model_param_distribution(
    lambda trial: build_model(trial=trial, kparams=kparams, show_summary=False),
    bits_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
    batch_size=64,
    n_trials=1000,
    save_path="nas_cnn1d+gnn_v0.0_model_param_distribution.png",
    figsize=(18, 6),
)